# Mestres do XGH — Aula 9
## Erros e debug: o programa que não morre

**Shift + Enter** roda a célula.

Hoje eu pago a dívida mais antiga do curso.

---
# 1. A dívida mais antiga do curso

Olhe a linha do tempo desta promessa:

| Quando | O que aconteceu |
|---|---|
| **Aula 2** | Alguém perguntou: *"e se a pessoa digitar `abc`?"* Eu disse: guardem |
| **Aula 3** | Resolvi **meio** com `.isdigit()`, e avisei que o jeito completo vinha depois |
| **Aula 8** | O `FileNotFoundError` mostrou que o problema é maior que o `abc` |
| **Aula 9** | Hoje |

Sete semanas. Vamos ver o que o `.isdigit()` resolvia e o que ele não resolvia:

In [ ]:
for texto in ["42", "abc", "-5", "3.14"]:
    print(f"{texto!r:8s} .isdigit() -> {texto.isdigit()}")

```
'42'     .isdigit() -> True
'abc'    .isdigit() -> False
'-5'     .isdigit() -> False    ← mas -5 É um número
'3.14'   .isdigit() -> False    ← e 3.14 também
```

Ele acerta o `abc` e erra o resto. Para funcionar de verdade, ele teria que saber sobre sinal
negativo, ponto decimal, espaço nas pontas, notação científica...

**A ferramenta certa é outra**, e é a de hoje.

---
# 2. Antes de tratar erro, aprenda a ler erro

Em oito aulas vocês viram muita tela vermelha. Hoje ela deixa de ser tela vermelha e vira
**informação**.

In [ ]:
def ler_nota(texto):
    return float(texto)

def processar(linha):
    nome, nota = linha.split()
    return nome, ler_nota(nota)

def main():
    return processar("Ana oito")

main()

```
Traceback (most recent call last):
  File "...", line 10, in <module>
    main()
  File "...", line 8, in main
    return processar("Ana oito")
  File "...", line 6, in processar
    return nome, ler_nota(nota)
  File "...", line 3, in ler_nota
    return float(texto)
ValueError: could not convert string to float: 'oito'
```

**Leia de baixo para cima.** Sempre.

1. **A última linha é o quê.** `ValueError: could not convert string to float: 'oito'` — já diz
   tudo: alguém tentou virar número um texto que não é número, e o texto era `'oito'`.
2. **A linha logo acima é o onde.** `in ler_nota`, no `float(texto)`. É ali que estourou.
3. **O resto é o caminho até lá.** `main` chamou `processar`, que chamou `ler_nota`. Serve para
   entender *como* o programa chegou naquele ponto.

> A palavra *traceback* quer dizer exatamente isso: o **rastro de volta**.
>
> Iniciante olha o traceback e vê um monte de linha. Quem já sofreu olha **a última linha
> primeiro**, e só sobe se precisar.

---
# 3. O catálogo: os erros que vocês já viram

Vocês colecionaram estes ao longo de oito aulas — sem perceber que estavam colecionando:

In [ ]:
casos = [
    ("NameError",         lambda: variavel_que_nao_existe),
    ("TypeError",         lambda: "2" + 2),
    ("ValueError",        lambda: int("abc")),
    ("IndexError",        lambda: [1, 2, 3][5]),
    ("KeyError",          lambda: {"a": 1}["b"]),
    ("ZeroDivisionError", lambda: 1 / 0),
    ("AttributeError",    lambda: "texto".apend("x")),
    ("FileNotFoundError", lambda: open("nao_existe.txt")),
]

for nome, f in casos:
    try:
        f()
    except Exception as e:
        print(f"{type(e).__name__:20s} {e}")

| Erro | Quer dizer | Onde vocês viram |
|---|---|---|
| `SyntaxError` | escrevi errado, nem chegou a rodar | Aula 3, o `if idade = 18` |
| `NameError` | esse nome não existe | Aula 6, escopo |
| `TypeError` | tipo errado para a operação | Aula 2, o `"2" + 2` |
| `ValueError` | tipo certo, valor impossível | hoje, o `int("abc")` |
| `IndexError` | posição que não existe | Aula 5, o `alunos[5]` |
| `KeyError` | chave que não existe | Aula 7 |
| `ZeroDivisionError` | dividiu por zero | ainda vai acontecer |
| `AttributeError` | esse objeto não tem esse método | quase sempre é erro de digitação |
| `FileNotFoundError` | o arquivo não está lá | Aula 8 |

> Repare na diferença entre os dois mais confundidos:
> - `TypeError` → **tipo errado**. `"2" + 2`: somar texto com número não existe.
> - `ValueError` → **tipo certo, valor impossível**. `int("abc")`: é texto, como deveria ser,
>   mas esse texto não vira número.

---
# 4. `try` / `except`

In [ ]:
try:
    numero = int("abc")
    print("consegui:", numero)
except ValueError:
    print("isso não é um número")

print("e o programa continua vivo")

Leia em português: **tente isto; se der aquele erro, faça aquilo em vez de morrer.**

```
try:
    (o que pode dar errado)
except ValueError:
    (o plano B, se der esse erro específico)
```

O ponto que importa: a última linha rodou. **O programa não morreu.**

## E agora, finalmente, o `abc`

In [ ]:
def pedir_numero(texto):
    try:
        return float(texto)
    except ValueError:
        return None


for t in ["42", "abc", "-5", "3.14", "", "  7  "]:
    print(f"{t!r:8s} -> {pedir_numero(t)}")

```
'42'     -> 42.0
'abc'    -> None
'-5'     -> -5.0      ← o .isdigit() errava este
'3.14'   -> 3.14      ← e este
''       -> None
'  7  '  -> 7.0       ← até o espaço nas pontas ele resolve
```

**Sete semanas depois, a dívida está paga.**

E repare no *porquê* de funcionar melhor: o `.isdigit()` tentava **prever** todas as formas de um
texto ser número. O `try/except` faz o contrário — **deixa o Python tentar** e só reage se falhar.

> Isso tem nome em programação: *"é mais fácil pedir perdão do que permissão"*.
>
> Em vez de checar mil condições antes, você tenta e trata a falha. Menos código, menos caso
> esquecido.

### 💥 Quebrando de propósito nº 1 — o `except` pelado

In [ ]:
def dividir(a, b):
    try:
        return a / b
    except:                       # ← sem dizer QUAL erro
        return "deu algum problema"


print(dividir(10, 2))
print(dividir(10, 0))
print(dividir(10, "dois"))
print(dividir(10, [1, 2]))

Parece que funcionou: nada quebrou. **E é exatamente esse o problema.**

O `except:` pelado engole **qualquer** erro — inclusive os que são bug seu. `dividir(10, [1,2])`
não é "um problema do usuário", é um erro de programação que você acabou de esconder de si mesmo.

Compare:

In [ ]:
def dividir_certo(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return "não dá para dividir por zero"


print(dividir_certo(10, 2))
print(dividir_certo(10, 0))

In [ ]:
print(dividir_certo(10, "dois"))

```
TypeError: unsupported operand type(s) for /: 'int' and 'str'
```

**Agora o bug aparece.** E aparecer é bom: é assim que você conserta.

> **Regra:** trate **o erro que você espera**, pelo nome. Erro que você não esperava deve
> continuar estourando — ele está te avisando de algo.
>
> `except:` pelado é a gambiarra mais cara do Python. Muito XGH, e você paga depois.

### 💥 Quebrando de propósito nº 2 — o `try` grande demais

In [ ]:
try:
    idade = int("30")
    dobro = idade * 2
    print("dobro:", dobro)
    nome = "Ana"
    print(nome.upper())
    print(nomee)                  # erro de digitação, escondido aqui dentro
except Exception as e:
    print("algo deu errado:", e)

Deu `name 'nomee' is not defined` — mas você teve que ler a mensagem para descobrir. Se o bloco
tivesse trinta linhas, boa sorte.

> **Envolva no `try` só a linha que pode falhar por um motivo que você já previu.** O resto fica
> de fora. `try` grande é `except` pelado disfarçado.

### `else` e `finally`

In [ ]:
def ler_arquivo(nome):
    try:
        f = open(nome, encoding="utf-8")
    except FileNotFoundError:
        print(f"não achei {nome}")
        return None
    else:
        conteudo = f.read()       # só roda se NÃO deu erro
        return conteudo
    finally:
        print("(passei por aqui de qualquer jeito)")


print(ler_arquivo("nao_existe.txt"))

- `else` → roda **só se o `try` deu certo**
- `finally` → roda **sempre**, com erro ou sem erro

O `finally` é para limpeza: fechar conexão, apagar arquivo temporário, liberar recurso. Foi
exatamente isso que o `with` da Aula 8 fez por vocês nos bastidores.

### `raise` — levantar o seu próprio erro

In [ ]:
def calcular_media(notas):
    if len(notas) == 0:
        raise ValueError("a lista de notas está vazia")
    return sum(notas) / len(notas)


print(calcular_media([7, 8, 9]))

In [ ]:
print(calcular_media([]))

Sem o `raise`, isso daria `ZeroDivisionError: division by zero` — tecnicamente correto e
inútil para quem chamou a função.

> **Um erro bom diz o que fazer.** `"a lista de notas está vazia"` é infinitamente melhor que
> `division by zero`, mesmo sendo o mesmo problema.
>
> Isso vale para os erros que vocês escrevem e é o que separa biblioteca boa de biblioteca ruim.

---
# 5. Depurar: achar o erro que não dá erro

`try/except` serve para o erro que **aparece**. O pior é o que não aparece — o do `elif` fora de
ordem na Aula 3, o do acumulador zerado na Aula 4, o da lista que virou `None` na Aula 5.

Três técnicas, em ordem de utilidade:

### 1. O `print` bem colocado

Não é vergonha. É a ferramenta mais usada do mundo, por todo mundo, inclusive por gente com
vinte anos de estrada.

In [ ]:
def media_dos_aprovados(notas, corte=7):
    total = 0
    quantos = 0
    for n in notas:
        print(f"  [debug] olhando {n}, corte {corte}, passa? {n >= corte}")
        if n >= corte:
            total += n
            quantos += 1
    print(f"  [debug] total={total} quantos={quantos}")
    return total / quantos


print(media_dos_aprovados([4, 8, 6, 9, 3, 7]))

> Marque seus prints com `[debug]`. Fica fácil de achar e de apagar depois.

### 2. Bisseção — cortar o problema no meio

Programa de 100 linhas dando resultado errado? Não leia as 100. Ponha um `print` na linha 50:

- **Valor certo na 50?** o problema está entre a 50 e a 100
- **Valor errado?** está entre a 1 e a 50

Repete. Sete cortes acham a agulha em 100 linhas — **é a busca binária do robô da Aula 6**, a
mesma ideia.

### 3. Ler o erro antes de mudar qualquer coisa

O reflexo de todo iniciante é começar a mexer no código. Não mexa. Leia a última linha do
traceback primeiro. Em metade das vezes ela já resolve.

---
# 6. Mão na massa — a agenda que não morre

Lembra do desafio da semana passada? *"Fazer o programa funcionar quando o `agenda.json` ainda
não existe."* Aqui está o jeito certo:

In [ ]:
import json


def carregar_agenda(arquivo="agenda.json"):
    try:
        with open(arquivo, encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"(primeira execução — criando {arquivo})")
        return []
    except json.JSONDecodeError:
        print(f"(o {arquivo} está corrompido — começando do zero)")
        return []


def salvar_agenda(agenda, arquivo="agenda.json"):
    with open(arquivo, "w", encoding="utf-8") as f:
        json.dump(agenda, f, indent=2, ensure_ascii=False)


agenda = carregar_agenda()
print("carregou:", agenda)

agenda.append({"nome": "Ana", "telefone": "11 90000-0001"})
salvar_agenda(agenda)

print("depois de salvar:", carregar_agenda())

Repare em **dois `except` diferentes** no mesmo `try`, cada um com uma mensagem própria:

- arquivo não existe → primeira execução, comece vazio
- arquivo existe mas está corrompido → avise e comece vazio

São dois problemas diferentes e merecem tratamento diferente. Um `except:` pelado trataria os
dois como "sei lá, deu ruim".

### 🔧 Exercício

Escreva `pedir_idade()` que:

1. pergunta a idade com `input()`
2. se não for número, avisa e **pergunta de novo** (um `while` com `try/except` dentro)
3. se for negativa ou maior que 130, avisa e pergunta de novo
4. só devolve quando o valor fizer sentido

É o `abc` da Aula 2, finalmente resolvido de verdade — e completo.

In [ ]:
# sua vez

---
# 💊 Pílula de IA — quando a IA erra, quem depura é você

## Etapa 1 — o modelo nunca levanta exceção

Vamos treinar o detector de spam da Aula 3 de novo e jogar coisas absurdas nele:

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

mensagens = [
    "promocao imperdivel clique aqui agora", "voce ganhou um premio gratis resgate ja",
    "credito pre aprovado clique no link", "gratis so hoje aproveite a promocao",
    "clique aqui e ganhe dinheiro rapido", "ultima chance promocao relampago clique",
    "voce foi sorteado resgate seu premio", "emprestimo liberado clique no link agora",
    "oi tudo bem vamos almocar amanha", "reuniao adiada para as 15h",
    "segue em anexo o relatorio do mes", "bom dia consegue revisar meu codigo hoje",
    "a aula de sabado comeca as 10h", "esqueci minha chave voce pode abrir",
    "parabens pelo trabalho ficou otimo", "vou chegar um pouco atrasado no almoco",
]
rotulos = ["spam"] * 8 + ["normal"] * 8

vetor = CountVectorizer()
modelo = MultinomialNB().fit(vetor.fit_transform(mensagens), rotulos)


def classificar(texto):
    p = modelo.predict_proba(vetor.transform([texto]))[0]
    i = p.argmax()
    return modelo.classes_[i], p[i] * 100


casos = [
    "promocao gratis clique aqui",
    "vamos almocar amanha",
    "o curso e gratis para os alunos",
    "asdfgh qwerty zxcvbn",
    "",
    "the quick brown fox jumps over the lazy dog",
    "1234567890",
]

for c in casos:
    r, conf = classificar(c)
    print(f"{r:6s} ({conf:5.1f}%)  <- {c!r}")

```
spam   ( 99.6%)  <- 'promocao gratis clique aqui'
normal ( 88.1%)  <- 'vamos almocar amanha'
spam   ( 61.2%)  <- 'o curso e gratis para os alunos'
normal ( 50.0%)  <- 'asdfgh qwerty zxcvbn'
normal ( 50.0%)  <- ''
normal ( 50.0%)  <- 'the quick brown fox jumps over the lazy dog'
normal ( 50.0%)  <- '1234567890'
```

Pare na quinta linha. **Eu mandei uma mensagem vazia e ele classificou.**

Mandei letras aleatórias: classificou. Mandei inglês, que ele nunca viu: classificou. Mandei
números: classificou.

**Nenhum `ValueError`. Nenhum traceback. Nenhuma tela vermelha.** Ele responde sempre.

## Etapa 2 — a diferença que muda o seu trabalho

| | Código | Modelo |
|---|---|---|
| Quando dá errado | **grita** — traceback, para tudo | **sussurra** — responde errado, educado |
| Como você descobre | na hora | só se você for olhar |
| Onde está o defeito | numa linha | espalhado nos dados e nos números |
| Como conserta | corrige a linha | dá mais exemplo (Aula 3) ou melhora o dado (Aula 8) |

> Um programa quebrado **te avisa**. Um modelo quebrado **te deixa confiante**.
>
> É por isso que testar IA é mais difícil que testar código: não existe traceback. Você tem que
> ir procurar.

## Etapa 3 — mas olhem aquele `50.0%`

Todas as absurdidades saíram com exatamente **50%**. Não é coincidência: 50% num problema de
duas classes é **cara ou coroa**. O modelo está dizendo, com o único vocabulário que tem:

> *"não faço a menor ideia."*

Ele **tem** a informação. Só ninguém estava olhando.

E lembra do `61%` da Aula 3, na frase "o curso e gratis para os alunos"? É o mesmo número, e é o
mesmo recado: ele está em cima do muro.

## Etapa 4 — juntando tudo o que vocês sabem

Agora dá para construir um classificador que **sabe dizer "não sei"** — usando o `if` da Aula 3,
a função da Aula 6 e o `raise` de hoje:

In [ ]:
def classificar_com_duvida(texto, limiar=80):
    if not texto.strip():
        raise ValueError("mensagem vazia")

    resultado, conf = classificar(texto)

    if conf < limiar:
        return "nao sei", conf
    return resultado, conf


for c in casos:
    try:
        r, conf = classificar_com_duvida(c)
        marca = "-> HUMANO" if r == "nao sei" else ""
        print(f"{r:8s} ({conf:5.1f}%) {marca:10s} <- {c!r}")
    except ValueError as e:
        print(f"{'ERRO':8s} {'':7s} {'':10s} <- {c!r}  ({e})")

```
spam     ( 99.6%)            <- 'promocao gratis clique aqui'
normal   ( 88.1%)            <- 'vamos almocar amanha'
nao sei  ( 61.2%) -> HUMANO  <- 'o curso e gratis para os alunos'
nao sei  ( 50.0%) -> HUMANO  <- 'asdfgh qwerty zxcvbn'
ERRO                         <- ''  (mensagem vazia)
nao sei  ( 50.0%) -> HUMANO  <- 'the quick brown fox jumps over the lazy dog'
nao sei  ( 50.0%) -> HUMANO  <- '1234567890'
```

**Todos os casos absurdos foram para revisão humana.** E o erro que ele cometia na Aula 3 — marcar
uma mensagem honesta como spam — virou um "não sei" em vez de uma resposta errada com cara de
certeza.

Isso tem nome na indústria: **human in the loop**. O modelo resolve o fácil, e o que ele não sabe
vai para uma pessoa.

E o `limiar` é um botão de negócio, não técnico:

In [ ]:
validos = []
for c in casos:
    if c.strip():
        validos.append(c)

for limiar in [60, 70, 80, 90, 95]:
    n = 0
    for c in validos:
        _, conf = classificar(c)
        if conf < limiar:
            n += 1
    print(f"limiar {limiar}% -> {n} de {len(validos)} casos para revisão humana")

```
limiar 60% -> 3 de 6 casos para revisão humana
limiar 70% -> 4 de 6 casos para revisão humana
limiar 80% -> 4 de 6 casos para revisão humana
limiar 90% -> 5 de 6 casos para revisão humana
limiar 95% -> 5 de 6 casos para revisão humana
```

Limiar alto = mais segurança, mais trabalho humano, mais custo.
Limiar baixo = mais automação, mais erro passando.

**Não existe resposta certa.** Existe a resposta certa *para aquele negócio*. Um filtro de spam de
e-mail pessoal e um sistema de triagem de exame médico não têm o mesmo limiar — e a conversa sobre
qual usar é com quem entende do problema, não com quem escreveu o código.

---

### O que fica da pílula de hoje

1. **Modelo não levanta exceção.** Ele responde sempre, inclusive sobre o que não conhece.
2. **Código grita, modelo sussurra.** Por isso testar IA dá mais trabalho que testar código.
3. **A confiança é a única tela vermelha que o modelo tem** — e transformar "confiança baixa" em
   "não sei" é uma decisão de engenharia que você escreve com um `if`.

---
# 🏠 Tarefa da semana

### 1. `pedir_numero()` de verdade
Uma função que insiste até receber um número válido, com mensagem clara a cada tentativa.
Use no jogo de adivinhação do Projeto 1 — ele quebra hoje se você digitar letra.

### 2. Agenda à prova de bala
Pegue a agenda da Aula 8 e trate: arquivo inexistente, arquivo corrompido, e o usuário digitando
coisa errada no menu. Cada caso com **seu próprio `except`**.

### 3. Leitor de CSV tolerante
Leia um CSV de notas em que **algumas linhas estão quebradas** (faltando coluna, nota com vírgula,
nota vazia). O programa deve pular a linha ruim, avisar qual foi, e continuar.

Crie o arquivo problemático você mesmo — inclusive é bom exercício pensar em como estragar dado.

### 4. Desafio
Pegue o `classificar_com_duvida` da pílula e escreva `revisar()`: quando o modelo diz "não sei",
o programa **pergunta para você** qual é a resposta certa e **acrescenta o exemplo no CSV**.

Rode algumas vezes, retreine, e veja o modelo melhorar com o que você ensinou.

*(Isso tem nome: **active learning**. E vocês vão ter construído um.)*

---

### 🤔 Para pensar até sábado

Na tarefa 3 você vai ler um CSV com `DictReader`, converter cada coluna na mão, tratar linha
quebrada, contar quantas passaram, somar, dividir.

Agora imagine que o arquivo tem **5.000 linhas** e **12 colunas**. E que você precisa da média por
cidade. E do total por mês. E dos 10 maiores.

Dá para fazer com `for`, `if` e dicionário — vocês já sabem tudo que é preciso.

**Quantas linhas de código isso vai ter?**

*(A resposta é a Aula 10: pandas. E ela é a última pílula separada do curso — a partir da 11, o
curso inteiro já é sobre IA.)*